# Model 1 — Dictionary-based Rule NER (Baseline)

**Proyek:** Deteksi Alergen pada Label Pangan Indonesia  
**Mata Kuliah:** COMP6885001 Natural Language Processing — BINUS University

---

## Deskripsi Model

Model 1 adalah **baseline rule-based** menggunakan kamus alergen (*allergen dictionary*) dengan algoritma **greedy longest-match**.

**Cara kerja:**
1. Teks bahan dipecah menjadi token (kata)
2. Untuk setiap posisi token, cari kecocokan kata/frase dalam kamus alergen
3. Frase terpanjang diprioritaskan (greedy)
4. Token yang cocok diberi label BIO: `B-{KATEGORI}` (awal) dan `I-{KATEGORI}` (lanjutan)

**Keunggulan:** Cepat, transparan, tidak perlu training  
**Kelemahan:** Hanya mendeteksi exact match, tidak memahami konteks, tidak bisa generalisasi ke kata baru

**Tugas:** Named Entity Recognition (NER) — token-level + category-level evaluation  
**Tidak perlu GPU** — dapat dijalankan di CPU lokal maupun Colab

In [ ]:
import re
import json
from collections import Counter, defaultdict

## 1. Kamus Alergen (ALLERGEN_CATEGORIES)

Kamus berisi ~150 keyword untuk 8 kategori alergen utama berdasarkan:
- **BPOM Indonesia** — regulasi pelabelan pangan
- **EU FIR No. 1169/2011** — standar alergen Eropa

Entri multi-kata (`tepung terigu`, `peanut butter`) ditempatkan **lebih awal** dalam list untuk memastikan greedy longest-match bekerja benar.

In [ ]:
ALLERGEN_CATEGORIES = {
    'GLUTEN': [
        'tepung terigu', 'tepung gandum', 'wheat starch', 'wheat bran', 'wheat germ',
        'malt extract', 'barley malt', 'hydrolyzed wheat protein',
        'tepung', 'terigu', 'gandum', 'gluten', 'barley', 'rye', 'oats',
        'sereal', 'roti', 'mie', 'pasta', 'biskuit', 'cracker', 'couscous',
        'bulgur', 'semolina', 'spelt', 'durum', 'kamut', 'malt', 'farina',
    ],
    'SUSU': [
        'susu sapi', 'milk powder', 'susu bubuk', 'nonfat milk', 'milk protein',
        'susu kental manis', 'whey protein', 'calcium caseinate', 'sodium caseinate',
        'rennet casein', 'milk solids', 'susu evaporasi', 'milk fat',
        'susu', 'keju', 'mentega', 'krim', 'yoghurt', 'butter', 'cream',
        'casein', 'kasein', 'whey', 'laktosa', 'lactose', 'skimmilk',
        'ghee', 'buttermilk', 'lactalbumin', 'lactoglobulin', 'dairy', 'caseinate',
    ],
    'TELUR': [
        'putih telur', 'kuning telur', 'egg white', 'egg yolk', 'egg powder',
        'telur bubuk', 'egg solids', 'egg protein', 'egg lecithin',
        'telur', 'egg', 'ovalbumin', 'ovomucoid', 'ovomucin',
        'lysozyme', 'mayonnaise', 'mayones', 'albumin', 'meringue',
    ],
    'KACANG': [
        'kacang tanah', 'peanut butter', 'kacang pohon', 'kacang mede', 'kacang mete',
        'kacang almond', 'kacang kenari', 'pine nut', 'brazil nut', 'tree nuts',
        'peanut', 'almond', 'cashew', 'walnut', 'pistachio', 'hazelnut',
        'pecan', 'macadamia', 'chestnut', 'groundnut', 'arachis',
    ],
    'KEDELAI': [
        'soy milk', 'susu kedelai', 'soy sauce', 'soy lecithin', 'soy protein',
        'soy isolate', 'soy flour', 'textured soy', 'lesitin kedelai', 'lesitin nabati',
        'soya lecithin',
        'kedelai', 'soy', 'soybean', 'tahu', 'tempe', 'tofu', 'tempeh',
        'edamame', 'miso', 'natto', 'yuba', 'okara',
        'lesitin', 'lecithin',
    ],
    'SEAFOOD': [
        'ikan teri', 'ikan tongkol', 'ikan kembung', 'ikan lele', 'ikan nila',
        'ikan bandeng', 'fish sauce', 'saus ikan', 'shrimp paste', 'fish oil',
        'krill oil', 'anchovy paste', 'saos tiram', 'saus tiram', 'oyster sauce',
        'ikan', 'tuna', 'salmon', 'sarden', 'makarel', 'cod', 'herring',
        'anchovy', 'udang', 'kerang', 'prawn', 'shrimp', 'cumi', 'sotong',
        'kepiting', 'crab', 'lobster', 'mussel', 'oyster', 'scallop',
        'abalone', 'terasi',
    ],
    'WIJEN': [
        'sesame oil', 'minyak wijen', 'biji wijen',
        'wijen', 'sesame', 'tahini',
    ],
    'SULFITE': [
        'sodium sulfite', 'potassium metabisulfite', 'sulfur dioxide',
        'natrium sulfit', 'kalium metabisulfit',
        'sulfite', 'sulfit',
    ],
}

ALL_CATEGORIES = list(ALLERGEN_CATEGORIES.keys())
print(f'Kategori alergen : {ALL_CATEGORIES}')
print(f'Total keyword    : {sum(len(v) for v in ALLERGEN_CATEGORIES.values())}')

## 2. BIO Labeler — Greedy Longest-Match

**BIO Tagging:**
- `B-CAT` — token *awal* entitas alergen (Beginning)
- `I-CAT` — token *lanjutan* frase alergen multi-kata (Inside)
- `O` — bukan alergen (Outside)

**Greedy longest-match:** Frase terpanjang dicocokkan lebih dulu. Contoh: `"peanut butter"` → `B-KACANG I-KACANG`, bukan `B-KACANG O`.

In [ ]:
def build_phrase_list(categories):
    phrases = []
    for cat, words in categories.items():
        for word in words:
            phrases.append((word.lower(), cat))
    phrases.sort(key=lambda x: len(x[0].split()), reverse=True)
    return phrases

PHRASE_LIST = build_phrase_list(ALLERGEN_CATEGORIES)

def tokenize_text(text):
    return re.findall(r'\w+|[^\w\s]', text.lower())

def bio_labeler(text):
    tokens = tokenize_text(text)
    labels = ['O'] * len(tokens)
    i = 0
    while i < len(tokens):
        matched = False
        for phrase, cat in PHRASE_LIST:
            phrase_tokens = tokenize_text(phrase)
            plen = len(phrase_tokens)
            if tokens[i:i + plen] == phrase_tokens:
                if all(labels[j] == 'O' for j in range(i, i + plen)):
                    labels[i] = f'B-{cat}'
                    for j in range(i + 1, i + plen):
                        labels[j] = f'I-{cat}'
                    i += plen
                    matched = True
                    break
        if not matched:
            i += 1
    return list(zip(tokens, labels))

def predict_categories(text):
    """Return sorted list of allergen categories detected in text."""
    results = bio_labeler(text)
    return sorted({label[2:] for _, label in results if label.startswith('B-')})

print('BIO labeler siap.')
print(f'Jumlah frase dalam kamus: {len(PHRASE_LIST)}')
print(f'Contoh 5 frase teratas (greedy order):')
for phrase, cat in PHRASE_LIST[:5]:
    print(f'  [{cat}] "{phrase}"')

## 3. Demo Inferensi

In [ ]:
DEMO_TEXTS = [
    'Komposisi: tepung terigu, gula, susu bubuk, kacang tanah, perisa alami.',
    'Bahan: tuna, shrimp paste, soy sauce, sesame oil, garam.',
    'Ingredients: peanut butter, milk powder, wheat starch, soy lecithin.',
    'Komposisi: gula, pati jagung, air, pewarna (E150a), tanpa alergen.',
    'Bahan: lesitin nabati, minyak sawit, coklat, susu kental manis, telur.',
]

for text in DEMO_TEXTS:
    print(f'Input : {text}')
    results = bio_labeler(text)
    allergen_tokens = [(t, l) for t, l in results if l != 'O']
    categories = predict_categories(text)
    
    if allergen_tokens:
        print(f'Token alergen terdeteksi:')
        for token, label in allergen_tokens:
            print(f'  {token:<22} → {label}')
        print(f'Kategori: {categories}')
    else:
        print(f'Tidak ada alergen terdeteksi.')
    print()

## 4. Evaluasi — Category-Level F1

Evaluasi menggunakan **50 produk pangan Indonesia** yang dianotasi manual (`evaluation_dataset.json`).

Format file evaluasi:
```json
[
  {
    "product_name": "Indomie_goreng.jpeg",
    "ingredient_text": "tepung terigu, gula, ...",
    "ground_truth": ["GLUTEN", "KEDELAI"]
  }
]
```

File ini dibuat dengan menggabungkan hasil OCR dan anotasi manual dari `nlp_eval_annotated.json`.

> **Catatan:** Jika file tidak ada, sel evaluasi dilewati dan hanya demo yang dijalankan.

In [ ]:
import os

EVAL_FILE = 'evaluation_dataset.json'

if not os.path.exists(EVAL_FILE):
    print(f'File {EVAL_FILE} tidak ditemukan.')
    print('Jalankan integrasi OCR + anotasi terlebih dahulu untuk membuat file ini.')
    print('Atau gunakan demo_evaluate() di bawah dengan data sampel.')
    EVAL_DATA = None
else:
    with open(EVAL_FILE, 'r', encoding='utf-8') as f:
        EVAL_DATA = json.load(f)
    print(f'Loaded {len(EVAL_DATA)} produk dari {EVAL_FILE}')

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

def evaluate_model(eval_data, predict_fn, all_categories):
    """
    Evaluasi model pada data gold label.
    predict_fn: function(text) -> list of category strings
    """
    y_true_raw, y_pred_raw = [], []
    for item in eval_data:
        pred = predict_fn(item['ingredient_text'])
        y_pred_raw.append(pred)
        y_true_raw.append(item['ground_truth'])

    mlb = MultiLabelBinarizer(classes=all_categories)
    Y_true = mlb.fit_transform(y_true_raw)
    Y_pred = mlb.transform(y_pred_raw)

    report = classification_report(Y_true, Y_pred, target_names=all_categories, zero_division=0)
    micro_p = precision_score(Y_true, Y_pred, average='micro', zero_division=0)
    micro_r = recall_score(Y_true, Y_pred, average='micro', zero_division=0)
    micro_f = f1_score(Y_true, Y_pred, average='micro', zero_division=0)
    macro_f = f1_score(Y_true, Y_pred, average='macro', zero_division=0)

    return report, micro_p, micro_r, micro_f, macro_f


if EVAL_DATA:
    print('=== MODEL 1: Dictionary-based NER ===')
    report, mp, mr, mf, macro_f = evaluate_model(EVAL_DATA, predict_categories, ALL_CATEGORIES)
    print(report)
    print(f'Micro  Precision : {mp:.4f}')
    print(f'Micro  Recall    : {mr:.4f}')
    print(f'Micro  F1        : {mf:.4f}')
    print(f'Macro  F1        : {macro_f:.4f}')

    # Simpan untuk perbandingan
    M1_RESULTS = {
        'model': 'Dictionary NER (Baseline)',
        'micro_precision': round(mp, 4),
        'micro_recall': round(mr, 4),
        'micro_f1': round(mf, 4),
        'macro_f1': round(macro_f, 4),
    }
    print(f'\nM1_RESULTS = {M1_RESULTS}')
else:
    print('Evaluasi dilewati — file evaluation_dataset.json tidak ada.')
    print('M1_RESULTS = None')
    M1_RESULTS = None

## 5. Analisis Hasil

### Kekuatan Model 1 (Dictionary NER)
- **Presisi tinggi** untuk keyword exact-match — ketika kata ada di kamus, hampir selalu benar
- **Transparan** — mudah di-debug dan diperbaiki
- **Cepat** — tidak ada training, langsung inference
- **Interpretable** — bisa jelaskan kenapa suatu token diklasifikasikan sebagai alergen

### Kelemahan Model 1 (Dictionary NER)
- **Recall rendah** — hanya mendeteksi kata yang **persis** ada di kamus
- Tidak mengenal sinonim, variasi ejaan, atau konteks semantik
- Tidak bisa generalisasi ke kata baru ("oat milk" vs "susu oat" vs "oats")
- Tidak memahami konteks: "tanpa kedelai" vs "kedelai" — keduanya detect KEDELAI
- Rentan terhadap OCR noise: "terigu" → "tergu" tidak terdeteksi

### Perbandingan dengan Model Lain
Model ini menjadi baseline untuk dibandingkan dengan:
- **Model 2** (TF-IDF + LR): classical ML, document-level
- **Model 3** (mBERT NER): deep learning multilingual
- **Model 4** (IndoBERT NER): deep learning bahasa Indonesia

In [ ]:
print('=== RINGKASAN MODEL 1 ===')
print()
print('Model     : Dictionary-based Rule NER (Greedy Longest-Match)')
print('Task      : NER + Category-level allergen detection')
print('Training  : Tidak perlu (rule-based)')
print('Keywords  :', sum(len(v) for v in ALLERGEN_CATEGORIES.values()))
print('Kategori  :', ALL_CATEGORIES)
print()
if M1_RESULTS:
    print('--- Hasil Evaluasi (Gold Label, 50 produk) ---')
    for k, v in M1_RESULTS.items():
        if k != 'model':
            print(f'  {k:<22}: {v}')
else:
    print('[Hasil evaluasi belum tersedia — jalankan dengan evaluation_dataset.json]')
print()
print('Keunggulan: Cepat, transparan, presisi tinggi untuk exact match')
print('Kelemahan : Recall rendah, tidak generalisasi ke kata baru/variasi ejaan')